Task 1: News Topic Classifier Using BERT

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score
import gradio as gr
import warnings
warnings.filterwarnings("ignore")

dataset = load_dataset("ag_news")

label_names = ["World", "Sports", "Business", "Sci/Tech"]

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_data = tokenized["train"].shuffle(seed=42).select(range(5000))
test_data = tokenized["test"].shuffle(seed=42).select(range(1000))

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./bert_news_model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

results = trainer.evaluate()
print(f"\nFinal Accuracy: {results['eval_accuracy']:.4f}")
print(f"Final F1 Score: {results['eval_f1']:.4f}")

model.save_pretrained("./bert_news_final")
tokenizer.save_pretrained("./bert_news_final")

loaded_model = BertForSequenceClassification.from_pretrained("./bert_news_final")
loaded_tokenizer = BertTokenizer.from_pretrained("./bert_news_final")
loaded_model.eval()

def predict_news(text):
    inputs = loaded_tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = loaded_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0]
    pred_label = label_names[torch.argmax(probs).item()]
    confidence = {label_names[i]: float(probs[i]) for i in range(4)}
    return pred_label, confidence

def gradio_predict(text):
    label, conf = predict_news(text)
    result = f"Predicted Category: {label}\n\nConfidence Scores:\n"
    for k, v in conf.items():
        result += f"  {k}: {v:.4f}\n"
    return result

interface = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=3, placeholder="Enter a news headline here..."),
    outputs=gr.Textbox(label="Prediction"),
    title="News Topic Classifier (BERT)",
    description="Classifies news headlines into: World, Sports, Business, or Sci/Tech",
    examples=[
        ["NASA launches new satellite to study climate change"],
        ["Stock markets fall amid inflation fears"],
        ["Manchester United wins Premier League title"],
        ["New AI model outperforms humans in medical diagnosis"]
    ]
)

interface.launch(share=True)